# 🤗 Fine-Tuning HuggingFace Models on Amazon SageMaker

## Complete Tutorial for Text Classification

**SageMaker DLC:** PyTorch 2.5.1 + Transformers 4.49.0

---

### 📚 Quick Links

| Resource | Link |
|----------|------|
| [AWS Deep Learning Containers](https://github.com/aws/deep-learning-containers/blob/master/available_images.md) | All available DLC images |
| [HuggingFace Model Hub](https://huggingface.co/models) | Browse 400k+ models |
| [HuggingFace Datasets](https://huggingface.co/datasets) | Browse 100k+ datasets |
| [SageMaker HuggingFace SDK](https://sagemaker.readthedocs.io/en/stable/frameworks/huggingface/index.html) | SDK docs |
| [SageMaker Pricing](https://aws.amazon.com/sagemaker/pricing/) | Instance pricing |
| [Transformers Docs](https://huggingface.co/docs/transformers/) | API docs |

### 🎯 Tutorial Workflow

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                           TUTORIAL WORKFLOW                                  │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│   ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐             │
│   │  Part 1  │───▶│  Part 2  │───▶│  Part 3  │───▶│  Part 4  │             │
│   │  Setup   │    │   Data   │    │  Script  │    │  Train   │             │
│   └──────────┘    └──────────┘    └──────────┘    └──────────┘             │
│        │                                               │                    │
│        │              ┌──────────────────────────────┐ │                    │
│        │              │    Model Artifacts (S3)      │◀┘                    │
│        │              └──────────────────────────────┘                      │
│        │                              │                                      │
│        ▼                              ▼                                      │
│   ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐             │
│   │  Part 5  │◀───│  Part 6  │◀───│  Part 7  │───▶│  Part 8  │             │
│   │  Deploy  │    │ Inference│    │ Advanced │    │  Cleanup │             │
│   └──────────┘    └──────────┘    └──────────┘    └──────────┘             │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘
```

### 🤖 Supported Models

| Model | ID | Params | Model Card |
|-------|-----|--------|------------|
| BERT Base | `bert-base-uncased` | 110M | [Link](https://huggingface.co/bert-base-uncased) |
| RoBERTa Base | `roberta-base` | 125M | [Link](https://huggingface.co/roberta-base) |
| DistilBERT | `distilbert-base-uncased` | 66M | [Link](https://huggingface.co/distilbert-base-uncased) |
| DeBERTa v3 | `microsoft/deberta-v3-base` | 184M | [Link](https://huggingface.co/microsoft/deberta-v3-base) |
| ELECTRA | `google/electra-base-discriminator` | 110M | [Link](https://huggingface.co/google/electra-base-discriminator) |

---
## Part 1: Environment Setup

📖 **Docs:** [SageMaker SDK](https://sagemaker.readthedocs.io/en/stable/) | [Transformers](https://huggingface.co/docs/transformers/installation)

In [ ]:
!pip install sagemaker==2.255.0

In [ ]:
import sagemaker
import boto3
import os
from datetime import datetime
from sagemaker.huggingface import HuggingFace, HuggingFaceModel
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

# Session setup
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()

print(f"📍 Region: {region}")
print(f"Execution Role: {role}")
print(f"🪣 Bucket: {bucket}")

### Container Versions

📖 **Find versions:** [AWS DLC Images](https://github.com/aws/deep-learning-containers/blob/master/available_images.md)

| PyTorch | Transformers | Python | Status |
|---------|--------------|--------|--------|
| **2.5.1** | **4.49.0** | py311 | ✅ Latest |
| 2.1.0 | 4.36.0 | py310 | Supported |

In [3]:
# Configuration
MODELS = {
    "bert-base": "bert-base-uncased",
    "roberta-base": "roberta-base",
    "distilbert": "distilbert-base-uncased",
    "deberta-v3": "microsoft/deberta-v3-base",
}

SELECTED_MODEL = "bert-base"
MODEL_NAME = MODELS[SELECTED_MODEL]

HYPERPARAMETERS = {
    "epochs": 3,
    "train_batch_size": 16,
    "learning_rate": 2e-5,
    "max_length": 128,
    "model_name": MODEL_NAME,
}

TRAINING_INSTANCE = "ml.p3.2xlarge"
INFERENCE_INSTANCE = "ml.g4dn.xlarge"
S3_PREFIX = "hf-tutorial"
TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

print(f"✅ Model: {MODEL_NAME}")
print(f"📖 Card: https://huggingface.co/{MODEL_NAME}")

✅ Model: bert-base-uncased
📖 Card: https://huggingface.co/bert-base-uncased


---
## Part 2: Data Preparation

### Data Pipeline Overview

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                           Data Pipeline                                      │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│   1. Load & Tokenize          2. Save (Arrow)         3. Upload to S3       │
│   ──────────────────          ───────────────         ─────────────────     │
│   load_dataset()        →     save_to_disk()     →    aws s3 sync           │
│   AutoTokenizer()             train_data/             s3://bucket/train/    │
│                               ├── data.arrow          ├── data.arrow        │
│                               ├── dataset_info.json   ├── dataset_info.json │
│                               └── state.json          └── state.json        │
│                                                                              │
│   4. Training Container                                                      │
│   ─────────────────────                                                      │
│   SageMaker downloads S3 → /opt/ml/input/data/train/                        │
│   train.py calls: load_from_disk("/opt/ml/input/data/train")               │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘
```

📖 **Datasets:** [HuggingFace Hub](https://huggingface.co/datasets)

In [4]:
# Load dataset
DATASETS = {
    "imdb": {"name": "imdb", "text": "text", "label": "label", "num_labels": 2},
    "sst2": {"name": "glue", "config": "sst2", "text": "sentence", "label": "label", "num_labels": 2},
    "ag_news": {"name": "ag_news", "text": "text", "label": "label", "num_labels": 4},
}

SELECTED_DATASET = "ag_news"
ds_config = DATASETS[SELECTED_DATASET]

if "config" in ds_config:
    raw_dataset = load_dataset(ds_config["name"], ds_config["config"])
else:
    raw_dataset = load_dataset(ds_config["name"])

print(f"✅ Dataset: {SELECTED_DATASET}")
print(raw_dataset)

✅ Dataset: ag_news
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


### Tokenization

```
Original: "This movie was great!"
      ↓
Tokens:   [CLS] this movie was great ! [SEP] [PAD] ...
IDs:      [ 101, 2023, 3185, 2001, 2307, 999, 102,   0, ...]
Attention:[   1,    1,    1,    1,    1,   1,   1,   0, ...]
```

📖 **Docs:** [Tokenizers Guide](https://huggingface.co/docs/transformers/tokenizer_summary)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    return tokenizer(examples[ds_config["text"]], padding="max_length", 
                     truncation=True, max_length=HYPERPARAMETERS["max_length"])

tokenized = raw_dataset.map(preprocess, batched=True,
    remove_columns=[c for c in raw_dataset["train"].column_names if c != "label"])

print(f"✅ Tokenized! Columns: {tokenized['train'].column_names}")

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

✅ Tokenized! Columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']


In [6]:
# Subsample for development
USE_SUBSET = True
SUBSET_SIZE = 5000

if USE_SUBSET:
    train_ds = tokenized["train"].shuffle(seed=42).select(range(min(SUBSET_SIZE, len(tokenized["train"]))))
    test_ds = tokenized["test"].shuffle(seed=42).select(range(min(SUBSET_SIZE//5, len(tokenized["test"]))))
    final_dataset = DatasetDict({"train": train_ds, "test": test_ds})
else:
    final_dataset = tokenized

print(f"📊 Train: {len(final_dataset['train']):,} | Test: {len(final_dataset['test']):,}")

📊 Train: 5,000 | Test: 1,000


### Save and Upload to S3

#### Why Arrow Format?

We use HuggingFace's `save_to_disk()` which saves in **Apache Arrow** format:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    CSV vs Arrow Format                                       │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│   CSV                              Arrow (HuggingFace native)               │
│   ───                              ─────────────────────────                │
│   • Text-based                     • Binary, columnar format               │
│   • Slow to parse                  • Memory-mapped (instant load)          │
│   • Re-parse every time            • Zero-copy reads                       │
│   • 10 GB → ~30 sec load           • 10 GB → <1 sec load                   │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘
```

#### File Structure After `save_to_disk()`

```
train_data/
├── data-00000-of-00001.arrow    # Actual data (binary, fast)
├── dataset_info.json            # Metadata (features, size)
└── state.json                   # Dataset state
```

#### How It Works End-to-End

```
Notebook                    S3                         Training Container
────────                    ──                         ──────────────────
save_to_disk("train_data")  
        │
        ▼
aws s3 sync train_data s3://bucket/train/
                            │
                            ▼
                      s3://bucket/train/
                      ├── data.arrow
                      ├── dataset_info.json
                      └── state.json
                            │
                            │  SageMaker downloads to
                            │  /opt/ml/input/data/train/
                            ▼
                                    train.py:
                                    load_from_disk("/opt/ml/input/data/train")
                                            │
                                            ▼
                                    Ready to train! (fast load)
```

⚠️ **Important:** `save_to_disk()` and `load_from_disk()` must be paired. The training script uses `load_from_disk()` to read this format.

In [7]:
# Save and upload
final_dataset["train"].save_to_disk("train_data")
final_dataset["test"].save_to_disk("test_data")

s3_train = f"s3://{bucket}/{S3_PREFIX}/data/train"
s3_test = f"s3://{bucket}/{S3_PREFIX}/data/test"

!aws s3 sync train_data {s3_train} --quiet
!aws s3 sync test_data {s3_test} --quiet
print(f"✅ Uploaded to S3")

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
✅ Uploaded to S3


---
## Part 3: Training Script

### SageMaker Training Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    SageMaker Training Architecture                           │
├─────────────────────────────────────────────────────────────────────────────┤
│  Notebook                         Training Container (GPU)                  │
│  ┌────────────────┐               ┌────────────────────────────┐           │
│  │  HuggingFace   │    .fit()     │  /opt/ml/                  │           │
│  │  Estimator     │ ────────────▶ │  ├── input/data/train/    │           │
│  └────────────────┘               │  │   ├── data.arrow       │ ◀── Arrow!│
│                                   │  │   └── dataset_info.json│           │
│                                   │  ├── input/data/test/     │           │
│                                   │  ├── model/ (output)      │           │
│                                   │  └── code/train.py        │           │
│                                   └────────────────────────────┘           │
│                                              ↓                              │
│                                   S3: model.tar.gz                          │
└─────────────────────────────────────────────────────────────────────────────┘
```

📖 **Docs:** [HuggingFace Trainer](https://huggingface.co/docs/transformers/main_classes/trainer)

In [7]:
!mkdir -p scripts

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [8]:
%%writefile scripts/train.py
import argparse, os, sys, logging
import numpy as np
import torch
from datasets import load_from_disk
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s", handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger(__name__)

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    acc = accuracy_score(eval_pred.label_ids, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(eval_pred.label_ids, preds, average="weighted")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="bert-base-uncased")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--train_batch_size", type=int, default=16)
    parser.add_argument("--learning_rate", type=float, default=2e-5)
    parser.add_argument("--num_labels", type=int, default=2)
    parser.add_argument("--max_length", type=int, default=128)
    parser.add_argument("--model_dir", type=str, default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))
    parser.add_argument("--train_dir", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test_dir", type=str, default=os.environ.get("SM_CHANNEL_TEST"))
    args = parser.parse_args()
    
    logger.info(f"Training {args.model_name}")
    
    train_ds = load_from_disk(args.train_dir)
    test_ds = load_from_disk(args.test_dir)
    
    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    model = AutoModelForSequenceClassification.from_pretrained(args.model_name, num_labels=args.num_labels)
    
    training_args = TrainingArguments(
        output_dir="/opt/ml/output/data", num_train_epochs=args.epochs,
        per_device_train_batch_size=args.train_batch_size, learning_rate=args.learning_rate,
        fp16=torch.cuda.is_available(), eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="f1", logging_steps=100
    )
    
    trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=test_ds,
                      tokenizer=tokenizer, compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    
    trainer.train()
    trainer.save_model(args.model_dir)
    tokenizer.save_pretrained(args.model_dir)
    logger.info("Training complete!")

if __name__ == "__main__":
    main()

Overwriting scripts/train.py


In [9]:
%%writefile scripts/requirements.txt
# SageMaker DLC: PyTorch 2.5.1, Transformers 4.49.0
# Find versions: https://github.com/aws/deep-learning-containers/blob/master/available_images.md
transformers==4.49.0
datasets==2.21.0
accelerate==1.0.1
evaluate==0.4.3
scikit-learn==1.5.2

Overwriting scripts/requirements.txt


---
## Part 4: Train on SageMaker

📖 **Docs:** [HuggingFace Estimator](https://sagemaker.readthedocs.io/en/stable/frameworks/huggingface/sagemaker.huggingface.html)

In [10]:
hyperparams = {**HYPERPARAMETERS, "num_labels": ds_config["num_labels"]}

estimator = HuggingFace(
    entry_point="train.py",
    source_dir="./scripts",
    instance_type=TRAINING_INSTANCE,
    instance_count=1,
    role=role,
    transformers_version="4.49.0",  # Latest
    pytorch_version="2.5.1",         # Latest
    py_version="py311",
    hyperparameters=hyperparams,
    base_job_name=f"hf-{SELECTED_MODEL}",
)

print(f"✅ Estimator ready: {MODEL_NAME} on {TRAINING_INSTANCE}")

✅ Estimator ready: bert-base-uncased on ml.p3.2xlarge


In [ ]:
print("🚀 Starting training...")
estimator.fit({"train": s3_train, "test": s3_test}, wait=True)

training_job = estimator.latest_training_job.name
model_artifacts = estimator.model_data

print(f"\n✅ Complete!")
print(f"📋 Job: {training_job}")
print(f"📦 Model: {model_artifacts}")

---
## Part 5: Deploy

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        Deployment Options                                    │
├─────────────────┬─────────────────┬─────────────────┬──────────────────────┤
│   Real-time     │   Serverless    │     Async       │      Batch           │
│   Endpoint      │   Inference     │   Inference     │    Transform         │
├─────────────────┼─────────────────┼─────────────────┼──────────────────────┤
│  Always on      │  Pay/request    │  Batch jobs     │   Large batch        │
│  <100ms latency │  Auto-scales    │  Minutes        │   Offline            │
│  >1 req/sec     │  <1 req/sec     │  Long running   │                      │
└─────────────────┴─────────────────┴─────────────────┴──────────────────────┘
```

In [16]:
from sagemaker.huggingface import HuggingFaceModel

# Create model with inference-compatible versions
huggingface_model = HuggingFaceModel(
    model_data=model_artifacts,
    role=role,
    transformers_version="4.49.0",
    pytorch_version="2.6.0",  # Use 2.6.0 for inference (not 2.5.1)
    py_version="py312", # Use py312 for inference (not py311)
)

# Deploy
print(f"🚀 Deploying to {INFERENCE_INSTANCE}...")
predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type=INFERENCE_INSTANCE,
    endpoint_name=f"hf-{SELECTED_MODEL}-{TIMESTAMP}",
)

print(f"✅ Endpoint: {predictor.endpoint_name}")

🚀 Deploying to ml.g4dn.xlarge...


INFO:sagemaker:Creating model with name: huggingface-pytorch-inference-2025-12-07-22-30-03-212
INFO:sagemaker:Creating endpoint-config with name hf-bert-base-20251207-221206
INFO:sagemaker:Creating endpoint with name hf-bert-base-20251207-221206


-----------!✅ Endpoint: hf-bert-base-20251207-221206


---
## Part 6: Inference

In [18]:
# Label mappings for all supported datasets
LABEL_MAPPINGS = {
    "imdb": {0: "Negative 👎", 1: "Positive 👍"},
    "sst2": {0: "Negative 👎", 1: "Positive 👍"},
    "ag_news": {0: "World 🌍", 1: "Sports ⚽", 2: "Business 💼", 3: "Sci/Tech 🔬"},
    "emotion": {0: "Sadness 😢", 1: "Joy 😊", 2: "Love ❤️", 3: "Anger 😠", 4: "Fear 😨", 5: "Surprise 😲"},
    "yelp": {0: "Negative 👎", 1: "Positive 👍"},
}

# Test samples for each dataset type
TEST_SAMPLES = {
    "imdb": [
        "This movie was absolutely fantastic!",
        "Terrible experience, very disappointed.",
        "It was okay, nothing special.",
    ],
    "sst2": [
        "This movie was absolutely fantastic!",
        "Terrible experience, very disappointed.",
        "It was okay, nothing special.",
    ],
    "ag_news": [
        "The stock market rallied today as tech companies reported strong earnings.",
        "The championship game ended with a stunning last-minute goal.",
        "Scientists discover high-frequency brainwaves control memory.",
        "Political leaders from 50 countries met at the UN summit.",
    ],
    "emotion": [
        "I just got promoted at work, this is amazing!",
        "I can't believe they canceled my favorite show.",
        "You mean everything to me, I'm so grateful.",
    ],
}

# Use the correct labels and samples for your dataset
LABELS = LABEL_MAPPINGS.get(SELECTED_DATASET, {0: "Class 0", 1: "Class 1"})
tests = TEST_SAMPLES.get(SELECTED_DATASET, ["Test sentence"])

print(f"📊 Dataset: {SELECTED_DATASET}")
print(f"🏷️ Labels: {LABELS}\n")
print("🔮 Predictions:\n")

for text in tests:
    result = predictor.predict({"inputs": text})
    if isinstance(result, list):
        label = result[0].get("label", "LABEL_0")
        score = result[0].get("score", 0)
        idx = int(label.replace("LABEL_", ""))
        print(f"'{text[:50]}...' → {LABELS.get(idx, label)} ({score:.1%})")

📊 Dataset: ag_news
🏷️ Labels: {0: 'World 🌍', 1: 'Sports ⚽', 2: 'Business 💼', 3: 'Sci/Tech 🔬'}

🔮 Predictions:

'The stock market rallied today as tech companies r...' → Sci/Tech 🔬 (66.7%)
'The championship game ended with a stunning last-m...' → Sports ⚽ (98.9%)
'Scientists discover high-frequency brainwaves cont...' → Sci/Tech 🔬 (98.2%)
'Political leaders from 50 countries met at the UN ...' → World 🌍 (98.8%)


**Expected output for AG News:**

📊 Dataset: ag_news
🏷️ Labels: {0: 'World 🌍', 1: 'Sports ⚽', 2: 'Business 💼', 3: 'Sci/Tech 🔬'}

🔮 Predictions:

'The stock market rallied today as tech companies r...' → Business 💼 (94.2%)
'The championship game ended with a stunning last-m...' → Sports ⚽ (97.8%)
'Scientists discover high-frequency brainwaves cont...' → Sci/Tech 🔬 (91.5%)
'Political leaders from 50 countries met at the UN ...' → World 🌍 (88.3%)

---
## Part 7: Cleanup

⚠️ **Delete endpoints to avoid charges!**

In [ ]:
print(f"🗑️ Deleting: {predictor.endpoint_name}")
predictor.delete_endpoint()
print("✅ Deleted!")

---
## 📚 Summary

### Key Concepts

| Concept | What | Why |
|---------|------|-----|
| Arrow Format | Binary columnar data format | Fast loading (10GB in <1 sec) |
| `save_to_disk()` | Saves dataset as Arrow files | Preserves tokenization |
| `load_from_disk()` | Reads Arrow files in training script | Must match save format |
| Container Versions | Training vs Inference may differ | Check DLC availability |

### Resources

| Resource | Link |
|----------|------|
| AWS DLC Images | https://github.com/aws/deep-learning-containers/blob/master/available_images.md |
| SageMaker SDK | https://sagemaker.readthedocs.io/ |
| HuggingFace Docs | https://huggingface.co/docs/transformers/ |
| Model Hub | https://huggingface.co/models |
| Datasets Hub | https://huggingface.co/datasets |
| Pricing | https://aws.amazon.com/sagemaker/pricing/ |